# 01 - Dataset Validation

Loads the fixed `data/processed/final_dataset.csv` (built upstream by
`notebooks/00_data_acquisition/`) and runs **non-destructive validation only**: required
columns present, unique `document_id`, non-empty text, valid labels (exactly USCIS / DMV /
SSA / IRS), duplicate count, class counts, stratified-split feasibility, and document-length
extremes.

This notebook never rewrites or cleans the source file. If a problem is found, it is
reported here and fixed upstream in `00_data_acquisition/05_build_dataset.ipynb` as a new
documented dataset version.

All validation logic lives in `src/newstart_ai/data/validation.py` -- this notebook only
calls it and displays results.

### Load configuration and the fixed dataset

**Purpose:** Make the reusable `newstart_ai` package importable, then load the project's
configuration and the raw dataset file.

**Why this step is necessary:** Every notebook in this project reads its settings (file
paths, column names, label list, split ratios, model hyperparameters) from the same
`configs/*.yaml` files through `load_settings()`, instead of hard-coding values in each
notebook. This keeps every notebook, and the future web API, working from one shared source
of truth -- a change to `configs/base.yaml` automatically applies everywhere.

**Inputs:** `configs/base.yaml` (dataset location and column names) and
`data/processed/final_dataset.csv` (the fixed, already-cleaned dataset built by the Phase 0
crawler notebooks in `00_data_acquisition/`).

**Output:** `settings` (a typed configuration object) and `df` (the raw dataset as a pandas
DataFrame), plus a preview of the first five rows.

**How to interpret the result:** The printed row count should match the known dataset size
(754 rows). `df.head()` is just a visual sanity check that the expected columns
(`document_id`, `text`, `agency`, ...) loaded correctly -- no validation has happened yet,
that's the next cell.

In [1]:
# Make the reusable src/newstart_ai package importable from inside notebooks/.
# Every notebook in this project does this the same way, so logic never has to be
# copy-pasted into individual cells.
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "src"))

from newstart_ai.config import load_settings
from newstart_ai.data.validation import load_dataset, validate_dataset

# load_settings() reads configs/base.yaml, bert.yaml, llm.yaml, and rag.yaml into one
# typed object, so every notebook and the future API share the same configuration values.
settings = load_settings()

# load_dataset() only reads the CSV -- it does not clean, rewrite, or drop any rows.
# Non-destructive loading matters because this is the one fixed research dataset: any
# real data problem must be fixed upstream (00_data_acquisition), not patched here.
df = load_dataset(settings)
print(f"Loaded {len(df)} rows from {settings.base.dataset.path}")
df.head()

Loaded 754 rows from data/processed/final_dataset.csv


,document_id,filename,filepath,agency,form_number,document_type,url,file_exists,text,page_count,text_length,extraction_success
0,1,i-765ws.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-765,form,NaN,True,Form I-765 Worksheet USCIS\r\nForm I-765WS\r\n...,1,1281,True
1,2,i-361instr.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-361,instructions,NaN,True,Instructions for Affidavit of\r\nFinancial Sup...,6,17876,True
2,3,i-508instr.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-508,instructions,NaN,True,"Instructions for Waiver of Certain Rights,\r\n...",4,9966,True
3,4,g-1566instr.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,G-1566,instructions,NaN,True,Instructions for Request for Certificate of No...,4,11552,True
4,5,i-129s.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-129,form,NaN,True,Nonimmigrant Petition Based on Blanket L Petit...,8,14082,True


### Run non-destructive validation checks

**Purpose:** Check the dataset for the specific problems the project design calls out
(missing columns, duplicate IDs, empty text, invalid labels, severe imbalance, and whether a
reliable stratified split is even possible), and collect the results into one structured
report.

**Why this step is necessary:** Before any splitting or training happens, we need
documented proof the dataset is usable -- silently training on a broken dataset (duplicate
rows, wrong labels) would invalidate every downstream result. Because this project treats
the dataset as fixed, `validate_dataset` only *reports* problems; it never rewrites `df`.

**Inputs:** `df` (the loaded dataset) and `settings` (for the expected column names and the
allowed label list).

**Output:** `report`, a `ValidationReport` object with counts, warnings, and
recommendations. `report.model_dump()` renders it as a plain dictionary so it's easy to
read here.

**How to interpret the result:** Look first at `required_columns_present`,
`document_id_column_unique`, and `valid_labels` -- these must all be true or the dataset has
a structural problem. `imbalance_ratio` and `stratified_split_feasible` describe how
balanced the classes are; a large ratio is expected here (IRS is a small minority class) and
is handled later with class-weighted loss, not by rejecting the dataset.

In [2]:
# validate_dataset() performs every check from the project design (required columns,
# unique IDs, empty text, valid labels, duplicates, class balance, and whether a
# stratified split is even mathematically possible) without modifying df in any way.
report = validate_dataset(df, settings)

# .model_dump() turns the Pydantic report object into a plain dict for easy inspection here.
report.model_dump()

{'row_count': 754,
 'required_columns_present': True,
 'missing_columns': [],
 'document_id_column_unique': True,
 'duplicate_document_id_count': 0,
 'empty_text_count': 0,
 'duplicate_text_count': 0,
 'valid_labels': True,
 'invalid_label_values': [],
 'class_counts': [{'label': 'DMV', 'count': 277, 'percentage': 36.74},
  {'label': 'USCIS', 'count': 256, 'percentage': 33.95},
  {'label': 'SSA', 'count': 198, 'percentage': 26.26},
  {'label': 'IRS', 'count': 23, 'percentage': 3.05}],
 'minimum_class_count': 23,
 'imbalance_ratio': 12.043478260869565,
 'stratified_split_feasible': True,
 'stratified_split_blockers': [],
 'text_length': {'mean': 15312.241379310344,
  'median': 7739.0,
  'minimum': 114,
  'maximum': 639219,
  'p95': 46745.85000000008},
 'warnings': ['Class imbalance ratio is 12.0x (majority/minority) -- class-weighted loss will apply during BERT training.'],
 'recommendations': ["Report macro F1 (not accuracy) as the primary metric, and flag the smallest class's per-clas

## Class counts and imbalance

### Tabulate class counts

**Purpose:** Turn the per-class counts already computed inside `report` into a small,
sorted table that's easy to read at a glance.

**Why this step is necessary:** The raw imbalance ratio (see the next cell) doesn't show
*which* class is small or by how much. A sorted table makes the imbalance concrete before
we rely on it later to decide whether class-weighted loss is needed for BERT.

**Inputs:** `report.class_counts`, the list of (label, count, percentage) already computed
by `validate_dataset`.

**Output:** A pandas DataFrame with one row per agency, sorted from most to least common.

**How to interpret the result:** DMV and USCIS should dominate, SSA is smaller, and IRS is
by far the smallest class -- a known, deliberate limitation of this dataset (see
`docs/BLUEPRINT.md`), not a bug in this notebook.

In [3]:
import pandas as pd

# Convert the report's per-class counts into a DataFrame purely for readable display --
# report itself already contains all of this information; nothing is recomputed here.
pd.DataFrame([c.model_dump() for c in report.class_counts]).sort_values("count", ascending=False)

,label,count,percentage
0,DMV,277,36.74
1,USCIS,256,33.95
2,SSA,198,26.26
3,IRS,23,3.05


### Check imbalance ratio and split feasibility

**Purpose:** Print the majority-to-minority class ratio and confirm whether a reliable
stratified train/validation/test split is possible given this class distribution.

**Why this step is necessary:** The research design commits to exactly one frozen
stratified split (created in notebook 03). Before committing to that split, we need to know
in advance whether every class has enough rows for a meaningful 64/16/20 split -- an
infeasible split would be a critical error, not just a warning.

**Inputs:** Values already computed inside `report` (`imbalance_ratio`,
`stratified_split_feasible`, `stratified_split_blockers`).

**Output:** Printed text -- no new object is created.

**How to interpret the result:** A large imbalance ratio (roughly 12x here) is expected and
acceptable; `configs/bert.yaml` already defines a threshold above which BERT training
automatically switches to class-weighted loss. `stratified_split_feasible: True` with no
blockers means it's safe to proceed to splitting in notebook 03, even though the smallest
class (IRS) will end up with only a handful of test documents.

In [4]:
# These numbers come straight from the validation report -- printing them here just
# makes the imbalance and split-feasibility decision visible in the notebook's output.
print(f"Imbalance ratio (majority/minority): {report.imbalance_ratio:.1f}x")
print(f"Stratified split feasible: {report.stratified_split_feasible}")
for blocker in report.stratified_split_blockers:
    print(" -", blocker)

Imbalance ratio (majority/minority): 12.0x
Stratified split feasible: True


## Text length

### Inspect document length statistics

**Purpose:** Show the mean, median, minimum, maximum, and 95th-percentile character length
of the documents in the dataset.

**Why this step is necessary:** BERT can only process a limited number of tokens per
document (512 for `bert-base-uncased`). Knowing the length distribution now -- before any
modeling decisions are made -- is what motivates the long-document strategy comparison in
notebook 04 (truncate to the first 512 tokens vs. sample the beginning, middle, and end of
long documents).

**Inputs:** Text lengths already computed inside `report.text_length` during validation.

**Output:** A plain dictionary of length statistics.

**How to interpret the result:** A large gap between the median and maximum length (explored
further with plots in notebook 02) signals that a meaningful fraction of documents are far
longer than BERT's token limit, so naive "first 512 tokens" truncation could silently
discard most of a long document's content.

In [5]:
# Character-length statistics were already computed during validation; this just displays them.
report.text_length.model_dump()

{'mean': 15312.241379310344,
 'median': 7739.0,
 'minimum': 114,
 'maximum': 639219,
 'p95': 46745.85000000008}

## Warnings and recommendations

### Review warnings and recommendations

**Purpose:** Print every warning and recommendation the validator produced, in plain
English.

**Why this step is necessary:** `has_critical_errors` (checked in the final cell) only
captures *structural* problems (missing columns, duplicate IDs, invalid labels). Warnings
capture *non-critical* observations -- like class imbalance or very long documents -- that
don't block the pipeline but do need to be carried forward as documented decisions rather
than silently ignored.

**Inputs:** `report.warnings` and `report.recommendations`, both plain lists of strings.

**Output:** Printed text.

**How to interpret the result:** Expect to see the class-imbalance warning (handled via
class-weighted loss later) and a recommendation to inspect the longest documents in
notebook 02 before choosing a long-document strategy. No warnings would mean the dataset
needs no special handling beyond what's already planned.

In [6]:
# Warnings flag non-critical issues (e.g. imbalance, extreme document lengths) that should
# be carried forward as documented decisions in later notebooks, not silently ignored.
print("Warnings:")
for w in report.warnings:
    print(" -", w)

print()
# Recommendations are the validator's suggested next actions based on what it found --
# they inform, but don't automatically trigger, choices made in later notebooks.
print("Recommendations:")
for r in report.recommendations:
    print(" -", r)

Warnings:
 - Class imbalance ratio is 12.0x (majority/minority) -- class-weighted loss will apply during BERT training.

Recommendations:
 - Report macro F1 (not accuracy) as the primary metric, and flag the smallest class's per-class metrics as statistically uncertain given its small test slice.
 - Inspect the longest records in 02_exploratory_data_analysis before choosing the long-document strategy -- extreme lengths may be legitimate long documents or upstream extraction artifacts (merged pages, OCR noise).


## Decision

The dataset must not proceed to splitting/training while `report.has_critical_errors` is
`True`. Non-critical warnings (imbalance, long documents) are carried forward as documented
decisions -- see `docs/BLUEPRINT.md` Section 6 -- not silently fixed here.

### Gate: stop here if validation found a critical error

**Purpose:** Enforce, with a hard assertion, that the notebook cannot silently continue if
a *critical* validation problem exists.

**Why this step is necessary:** This is the single checkpoint that prevents a structurally
broken dataset (missing columns, duplicate IDs, invalid labels, or an infeasible split) from
ever reaching the splitting and training notebooks. Non-critical issues (imbalance, long
documents) are allowed through, since they're handled later by design, not treated as
blockers.

**Inputs:** `report.has_critical_errors`, a boolean computed from the structural checks
above.

**Output:** Either an `AssertionError` (which would stop the notebook) or a printed
confirmation message.

**How to interpret the result:** Seeing the "Validation passed" message means it's safe to
proceed to `02_exploratory_data_analysis.ipynb`. This assertion passing is itself part of
the research record -- documented evidence that the dataset met the minimum bar before any
further work was done on it.

In [7]:
# Hard stop: if any structural problem was found (missing columns, duplicate IDs, invalid
# labels, or an infeasible split), do not let the notebook continue silently.
assert not report.has_critical_errors, "Critical validation errors found -- fix upstream before continuing."
print("Validation passed: no critical errors. Proceed to 02_exploratory_data_analysis.ipynb.")

Validation passed: no critical errors. Proceed to 02_exploratory_data_analysis.ipynb.
